In [1]:

import librosa
import numpy as np


# ============================================================
# AUDIO CONFIGURATION
# ============================================================

TARGET_SR = 16000
# Standardize every audio file to 16 kHz.
# This matches the sample rate used by your preprocessing pipeline.

N_MELS = 128
# Number of Mel-frequency bands.
# This becomes the HEIGHT of the spectrogram:
#     128 frequency features

N_FFT = 1024
# Number of samples used for each FFT window.
# Larger values give better frequency resolution.

HOP_LENGTH = 256
# Number of samples between consecutive FFT windows.
# Smaller values give more time frames.


# ============================================================
# LOAD AUDIO
# ============================================================

def load_audio(audio_path, sr=TARGET_SR):

    audio, _ = librosa.load(
        str(audio_path),
        sr=sr,
        mono=True
    )
    # Load the audio as:
    # - mono
    # - 16 kHz
    #
    # Example:
    # stereo 44.1 kHz WAV
    #          ↓
    # mono 16 kHz waveform

    return audio


# ============================================================
# MEL-SPECTROGRAM EXTRACTION
# ============================================================

def extract_mel_spectrogram(audio):

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )
    # Convert the raw waveform into a Mel spectrogram.
    #
    # Output shape:
    #
    #     (128, number_of_time_frames)
    #
    # 128 = Mel-frequency bands
    # time_frames = how the audio changes over time

    mel_db = librosa.power_to_db(
        mel,
        ref=np.max
    )
    # Convert power values into decibels.
    #
    # This produces a log-scaled representation that is
    # generally more useful for neural-network audio models.
    #
    # ref=np.max means the loudest point is treated as 0 dB.

    return mel_db


# ============================================================
# FEATURE NORMALIZATION
# ============================================================

def normalize_features(features):

    mean = np.mean(features)
    # Calculate the average value of the entire spectrogram.

    std = np.std(features)
    # Calculate the standard deviation.

    features = (
        features - mean
    ) / (std + 1e-8)
    # Standardize the spectrogram:
    #
    # normalized = (value - mean) / standard deviation
    #
    # 1e-8 prevents division by zero.

    return features


# ============================================================
# COMPLETE FEATURE EXTRACTION
# ============================================================

def extract_features(audio_path):

    audio = load_audio(audio_path)
    # Step 1:
    # WAV → normalized sample rate → mono waveform

    features = extract_mel_spectrogram(
        audio
    )
    # Step 2:
    # waveform → Mel spectrogram → dB spectrogram

    features = normalize_features(
        features
    )
    # Step 3:
    # Normalize the spectrogram values.

    return features


# ============================================================
# FIXED-SIZE SPECTROGRAM
# ============================================================

def pad_or_crop(
    features,
    target_frames=400
):
    """
    Make every spectrogram the same size.

    This allows batches to be created by PyTorch.
    """

    current_frames = features.shape[1]
    # features.shape:
    #
    #     [128, time_frames]
    #
    # shape[1] gives us the number of time frames.

    if current_frames < target_frames:

        padding = target_frames - current_frames
        # Calculate how many additional frames are needed.

        features = np.pad(
            features,
            (
                (0, 0),
                (0, padding)
            ),
            mode="constant"
        )
        # Add zeros to the RIGHT side of the spectrogram.
        #
        # Frequency dimension:
        #     (0, 0) → don't change the 128 Mel bands
        #
        # Time dimension:
        #     (0, padding) → add frames at the end
        #
        # Result:
        #     [128, 400]

    else:

        features = features[
            :,
            :target_frames
        ]
        # If the audio is too long, keep only the
        # first 400 time frames.
        #
        # Result:
        #     [128, 400]

    return features


# ============================================================
# FINAL FEATURE EXTRACTION FUNCTION
# ============================================================

def extract_fixed_features(audio_path):

    features = extract_features(
        audio_path
    )
    # WAV
    #   ↓
    # waveform
    #   ↓
    # Mel spectrogram
    #   ↓
    # dB
    #   ↓
    # normalized features

    features = pad_or_crop(
        features
    )
    # Make every sample exactly:
    #
    #     [128, 400]

    return features.astype(
        np.float32
    )
    # Convert from NumPy's default floating-point type
    # to float32.
    #
    # PyTorch models normally use float32 tensors.

